> **Portfolio Version:** This notebook has been prepared for portfolio purposes. Confidential data and source files have been removed.


In [ ]:
# =====================================
# Sprint 2 - Star Schema Build
# =====================================

import pandas as pd
import numpy as np
import os
from pathlib import Path


In [ ]:
# =====================================
# Step 1 - Load Cleaned Staging Table
# =====================================

PROJECT_PATH = Path("..")

OUTPUT_PATH = PROJECT_PATH / "01_Data" / "processed"

stg_ndt_inspection = pd.read_csv(OUTPUT_PATH / "stg_ndt_inspection.csv")



In [ ]:
stg_ndt_inspection.info()
stg_ndt_inspection.head()

In [ ]:
print("=" * 60)
print("MISSING DATE CHECK")
print("=" * 60)

print(
    stg_ndt_inspection[
        ["RequestDate", "JDRDate"]
    ].isna().sum()
)

In [ ]:

os.getcwd() #to check the current directory

In [ ]:
os.listdir() # list of all files

In [ ]:
stg_ndt_inspection["RequestDate"] = pd.to_datetime(stg_ndt_inspection["RequestDate"], errors="coerce")
stg_ndt_inspection["JDRDate"] = pd.to_datetime(stg_ndt_inspection["JDRDate"], errors="coerce")

text_cols = ["Discipline", "Module", "Method", "RequestNo"]

for col in text_cols:
    stg_ndt_inspection[col] = stg_ndt_inspection[col].astype("string")

In [ ]:
stg_ndt_inspection.info()

In [ ]:
# =====================================
# Step 3 - Create Method Dimension
# =====================================

dim_method = (
    stg_ndt_inspection[["Method"]]
    .drop_duplicates()
    .sort_values("Method")
    .reset_index(drop=True)
)

dim_method.insert(0, "MethodKey", range(1, len(dim_method) + 1))

dim_method

In [ ]:
# =====================================
# Step 3A - Add Method Description
# =====================================

method_description_map = {
    "DPT": "Dye Penetrant Testing",
    "HT": "Hardness Testing",
    "MPT": "Magnetic Particle Testing",
    "MT": "Magnetic Testing",
    "PAUT": "Phased Array Ultrasonic Testing",
    "PMI": "Positive Material Identification",
    "PT": "Penetrant Testing",
    "RT": "Radiographic Testing",
    "UT": "Ultrasonic Testing"
}

dim_method["MethodDescription"] = dim_method["Method"].map(method_description_map)

dim_method

In [ ]:
dim_method[dim_method["MethodDescription"].isna()]

In [ ]:
# =====================================
# Step 3B - Add Method Category
# =====================================

method_category_map = {
    "DPT": "Surface Testing",
    "HT": "Material Verification",
    "MPT": "Surface Testing",
    "MT": "Surface Testing",
    "PAUT": "Volumetric Testing",
    "PMI": "Material Identification",
    "PT": "Surface Testing",
    "RT": "Volumetric Testing",
    "UT": "Volumetric Testing"
}

dim_method["MethodCategory"] = dim_method["Method"].map(method_category_map)

dim_method

In [ ]:
# =====================================
# Step 4 - Save Method Dimension
# =====================================

dim_method.to_csv(
    OUTPUT_PATH / "dim_method.csv",
    index=False
)

In [ ]:
# After saving, you can check:
OUTPUT_PATH

In [ ]:
# List all files in the Output folder:

list(OUTPUT_PATH.iterdir())

In [ ]:
# =====================================
# Step 5 - Create Discipline Dimension
# =====================================

dim_discipline = (
    stg_ndt_inspection[["Discipline"]]
    .drop_duplicates()
    .sort_values("Discipline")
    .reset_index(drop=True)
)

dim_discipline.insert(0, "DisciplineKey", range(1, len(dim_discipline) + 1))

dim_discipline

In [ ]:
# =====================================
# Step 6 - Save Discipline Dimension
# =====================================

dim_discipline.to_csv(
    OUTPUT_PATH / "dim_discipline.csv",
    index=False
)

In [ ]:
list(OUTPUT_PATH.iterdir())

In [ ]:
# =====================================
# Step 7 - Create Module Dimension
# =====================================

dim_module = (
    stg_ndt_inspection[["Module"]]
    .drop_duplicates()
    .sort_values("Module")
    .reset_index(drop=True)
)

dim_module.insert(0, "ModuleKey", range(1, len(dim_module) + 1))

dim_module

In [ ]:
# =====================================
# Step 8 - Save Module Dimension
# =====================================

dim_module.to_csv(
    OUTPUT_PATH / "dim_module.csv",
    index=False
)

In [ ]:
(OUTPUT_PATH / "dim_module.csv").exists()

In [ ]:
# =====================================
# Step 9 - Create Calendar Dimension
# =====================================

min_date = min(
    stg_ndt_inspection["RequestDate"].min(),
    stg_ndt_inspection["JDRDate"].min()
)

max_date = max(
    stg_ndt_inspection["RequestDate"].max(),
    stg_ndt_inspection["JDRDate"].max()
)

min_date, max_date

In [ ]:
stg_ndt_inspection["RequestDate"].head(10)

In [ ]:
stg_ndt_inspection["JDRDate"].head(10)

In [ ]:
# =====================================
# Step 9B - Create Calendar Dimension
# =====================================

calendar_start = "2023-01-01"
calendar_end = "2026-12-31"

dim_calendar = pd.DataFrame({
    "Date": pd.date_range(
        start=calendar_start,
        end=calendar_end,
        freq="D"
    )
})

dim_calendar.head()

In [ ]:
dim_calendar.tail()


In [ ]:
dim_calendar.shape

In [ ]:
# =====================================
# Step 9C - Add Calendar Attributes
# =====================================

dim_calendar["DateKey"] = dim_calendar["Date"].dt.strftime("%Y%m%d").astype(int)

dim_calendar["Year"] = dim_calendar["Date"].dt.year

dim_calendar["Quarter"] = "Q" + dim_calendar["Date"].dt.quarter.astype(str)

dim_calendar["MonthNumber"] = dim_calendar["Date"].dt.month

dim_calendar["MonthName"] = dim_calendar["Date"].dt.month_name()

dim_calendar["MonthYear"] = dim_calendar["Date"].dt.strftime("%b-%Y")

dim_calendar["DayOfMonth"] = dim_calendar["Date"].dt.day

dim_calendar["DayOfWeek"] = dim_calendar["Date"].dt.dayofweek + 1

dim_calendar["Weekday"] = dim_calendar["Date"].dt.day_name()

dim_calendar["IsWeekend"] = dim_calendar["DayOfWeek"].apply(
    lambda x: "Yes" if x >= 6 else "No"
)

dim_calendar = dim_calendar[
    [
        "DateKey",
        "Date",
        "Year",
        "Quarter",
        "MonthNumber",
        "MonthName",
        "MonthYear",
        "DayOfMonth",
        "DayOfWeek",
        "Weekday",
        "IsWeekend"
    ]
]

dim_calendar.head()

In [ ]:
# =====================================
# Step 9D - Save Calendar Dimension
# =====================================

dim_calendar.to_csv(
    OUTPUT_PATH / "dim_calendar.csv",
    index=False
)

In [ ]:
(OUTPUT_PATH / "dim_calendar.csv").exists()

In [ ]:
print(stg_ndt_inspection.columns.tolist())

In [ ]:
# =====================================
# Add MethodKey to Staging Table
# =====================================

stg_ndt_inspection = stg_ndt_inspection.merge(
    dim_method[["MethodKey", "Method"]],
    on="Method",
    how="left"
)

print(
    stg_ndt_inspection[
        ["Method", "MethodKey"]
    ].head()
)

print(
    "\nMissing MethodKey:",
    stg_ndt_inspection["MethodKey"].isna().sum()
)

In [ ]:
print(
    stg_ndt_inspection[
        ["Method", "MethodKey"]
    ]
    .drop_duplicates()
    .sort_values("MethodKey")
)

In [ ]:
stg_ndt_inspection["MethodKey"].isna().sum()

In [ ]:
# =====================================
# Add DisciplineKey to Staging Table
# =====================================

stg_ndt_inspection = stg_ndt_inspection.merge(
    dim_discipline[["DisciplineKey", "Discipline"]],
    on="Discipline",
    how="left"
)

print(
    stg_ndt_inspection[
        ["Discipline", "DisciplineKey"]
    ].head()
)

print(
    "\nMissing DisciplineKey:",
    stg_ndt_inspection["DisciplineKey"].isna().sum()
)

In [ ]:
print(
    stg_ndt_inspection[
        ["Discipline", "DisciplineKey"]
    ]
    .drop_duplicates()
    .sort_values("DisciplineKey")
)

In [ ]:
# =====================================
# Add ModuleKey to Staging Table
# =====================================

stg_ndt_inspection = stg_ndt_inspection.merge(
    dim_module[["ModuleKey", "Module"]],
    on="Module",
    how="left"
)

print(
    stg_ndt_inspection[
        ["Module", "ModuleKey"]
    ].head()
)

print(
    "\nMissing ModuleKey:",
    stg_ndt_inspection["ModuleKey"].isna().sum()
)

In [ ]:
print(
    stg_ndt_inspection[
        ["Module", "ModuleKey"]
    ]
    .drop_duplicates()
    .sort_values("ModuleKey")
)

In [ ]:
# =====================================
# Step 10E - Create Date Keys
# =====================================

stg_ndt_inspection["RequestDateKey"] = (
    stg_ndt_inspection["RequestDate"]
    .dt.strftime("%Y%m%d")
    .astype("Int64")
)

stg_ndt_inspection["JDRDateKey"] = (
    stg_ndt_inspection["JDRDate"]
    .dt.strftime("%Y%m%d")
    .astype("Int64")
)

In [ ]:
stg_ndt_inspection[["RequestDate", "RequestDateKey", "JDRDate", "JDRDateKey"]].head()

In [ ]:
# =====================================
# Step 10F - Validate Date Keys
# =====================================

valid_date_keys = set(dim_calendar["DateKey"])

stg_ndt_inspection["IsRequestDateValid"] = stg_ndt_inspection["RequestDateKey"].isin(valid_date_keys)

stg_ndt_inspection["IsJDRDateValid"] = stg_ndt_inspection["JDRDateKey"].isin(valid_date_keys)

print("=" * 60)
print("DATE KEY VALIDATION")
print("=" * 60)

print(
    stg_ndt_inspection[
        ["IsRequestDateValid", "IsJDRDateValid"]
    ].value_counts()
)

print("\nInvalid RequestDateKeys:",
      (~stg_ndt_inspection["IsRequestDateValid"]).sum())

print("Invalid JDRDateKeys:",
      (~stg_ndt_inspection["IsJDRDateValid"]).sum())

In [ ]:
# =====================================
# Step 10A - Create Fact Base Table
# =====================================

fact_ndt_inspection = stg_ndt_inspection[
    [
        "InspectionID",
        "RequestNo",
        "RequestDateKey",
        "JDRDateKey",
        "DisciplineKey",
        "ModuleKey",
        "MethodKey",
        "RequestedQty",
        "CompletedQty",
        "JDRQty"
    ]
].copy()

print("Fact rows:", len(fact_ndt_inspection))

print(
    fact_ndt_inspection[
        ["RequestDateKey", "JDRDateKey"]
    ].isna().sum()
)

In [ ]:
print(fact_ndt_inspection.shape)

print(
    fact_ndt_inspection[
        fact_ndt_inspection["InspectionID"].isin([2470, 3479])
    ]
)

In [ ]:
print("=" * 60)
print("FACT TABLE VALIDATION")
print("=" * 60)

print("Rows:", len(fact_ndt_inspection))

print("\nDuplicate InspectionID:")
print(fact_ndt_inspection["InspectionID"].duplicated().sum())

print("\nMissing Keys:")
print(
    fact_ndt_inspection[
        [
            "RequestDateKey",
            "JDRDateKey",
            "DisciplineKey",
            "ModuleKey",
            "MethodKey"
        ]
    ].isna().sum()
)

print("\nMissing Quantities:")
print(
    fact_ndt_inspection[
        [
            "RequestedQty",
            "CompletedQty",
            "JDRQty"
        ]
    ].isna().sum()
)

In [ ]:
# =====================================
# Step 10I - Save Fact Table
# =====================================

fact_ndt_inspection.to_csv(
    OUTPUT_PATH / "fact_ndt_inspection.csv",
    index=False
)

print("=" * 60)
print("FACT TABLE EXPORTED")
print("=" * 60)

print("Rows:", len(fact_ndt_inspection))
print("Columns:", len(fact_ndt_inspection.columns))

In [ ]:
print("=" * 60)
print("DIMENSION TABLES EXPORTED")
print("=" * 60)

print("dim_calendar rows:", len(dim_calendar))
print("dim_discipline rows:", len(dim_discipline))
print("dim_module rows:", len(dim_module))
print("dim_method rows:", len(dim_method))

In [ ]:
valid_date_keys = set(dim_calendar["DateKey"].astype(int))

missing_request_keys = sorted(
    set(
        fact_ndt_inspection["RequestDateKey"]
        .dropna()
        .astype(int)
    ) - valid_date_keys
)

missing_jdr_keys = sorted(
    set(
        fact_ndt_inspection["JDRDateKey"]
        .dropna()
        .astype(int)
    ) - valid_date_keys
)

print("Missing RequestDateKeys:", missing_request_keys)
print("Missing JDRDateKeys:", missing_jdr_keys)
print("Fact shape:", fact_ndt_inspection.shape)

In [ ]:
print(fact_ndt_inspection.shape)